[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/6_optimizacion/_assets/01_impresora_lineal.ipynb)

# Notebook 1 · La impresora

Acompaña a la **clase 1** de la unidad de modelado y optimización. Da por leídas
las cinco páginas.

Las páginas se leen; esto se mueve. Aquí vas a **arrastrar los precios y ver
saltar la respuesta**, subir y bajar la energía hasta encontrar el umbral con la
mano, y romper el problema a propósito de las cuatro maneras en que se puede
romper.

Al final hay una historia nueva —el invernadero— que **no está resuelta en
ninguna página**. Ése es el ejercicio. No hay nada que entregar.

Corre las celdas en orden con `Shift + Enter`.


In [ ]:
# === Celda 1 · Preparación =====================================
# La convención de signos, que es la fuente número uno de errores al pasar del
# papel al código. Es la misma tabla de la página 5:
#
#   - En las notas la constante va a la derecha; se admiten <=, >= y =.
#   - Al solver se le entrega TODO con <=:  a·x >= b  se escribe  -a·x <= -b.
#   - linprog MINIMIZA. Un máximo se resuelve con -c, y al valor que devuelve
#     hay que cambiarle el signo OTRA VEZ.
#   - Las cotas simples sobre una variable van en `bounds`, no como fila de A.

import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.optimize import linprog

try:
    from ipywidgets import interact, FloatSlider
    HAY_WIDGETS = True
except ImportError:
    HAY_WIDGETS = False

print('listo. widgets:', HAY_WIDGETS)


In [ ]:
# === Celda 2 · Las tres funciones que se usan todo el notebook ==

def resolver(c, A, b, bounds=None):
    """Maximiza c·x sujeto a A x <= b, x >= 0. Devuelve (x, valor, estado)."""
    r = linprog(c=-np.asarray(c, float), A_ub=np.asarray(A, float),
                b_ub=np.asarray(b, float), bounds=bounds, method='highs')
    if r.status != 0:                       # 2 = infactible, 3 = no acotado
        return None, None, r.message.split('.')[0]
    return r.x, -r.fun, 'óptimo'            # ojo con el signo de vuelta


def esquinas(A, b):
    """Las esquinas de la región, con el método de la página 3: cruzar las
    rectas de dos en dos y quedarse solo con los cruces que cumplen todo."""
    A, b = np.asarray(A, float), np.asarray(b, float)
    filas = np.vstack([A, [-1, 0], [0, -1]])
    lados = np.concatenate([b, [0, 0]])
    puntos = []
    for i, j in combinations(range(len(filas)), 2):
        M = filas[[i, j]]
        if abs(np.linalg.det(M)) < 1e-9:            # rectas paralelas
            continue
        p = np.linalg.solve(M, lados[[i, j]])
        if np.all(filas @ p <= lados + 1e-9):       # ¿cae dentro de lo demás?
            puntos.append(p)
    if not puntos:
        return np.empty((0, 2))
    P = np.unique(np.round(puntos, 9), axis=0) + 0.0
    P[np.abs(P) < 1e-9] = 0.0                        # quita los -0. del solve
    centro = P.mean(axis=0)
    return P[np.argsort(np.arctan2(*(P - centro).T[::-1]))]


def dibujar(c, A, b, nombres=None, titulo='', lim=11, ax=None, niveles=3):
    """La región, las rectas de cada restricción y la curva de nivel ganadora."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6.2, 6.2))
    c, A, b = np.asarray(c, float), np.asarray(A, float), np.asarray(b, float)
    nombres = nombres or [f'restricción {k+1}' for k in range(len(b))]
    V = esquinas(A, b)
    if len(V):
        ax.fill(V[:, 0], V[:, 1], alpha=.22, color='tab:purple', zorder=0,
                label='lo que se puede hacer')
    t = np.linspace(0, lim, 200)
    for k in range(len(b)):
        if A[k, 1]:
            ax.plot(t, (b[k] - A[k, 0] * t) / A[k, 1], lw=1.5, label=nombres[k])
        else:
            ax.axvline(b[k] / A[k, 0], lw=1.5, label=nombres[k])
    x, z, estado = resolver(c, A, b)
    if x is not None:
        for k in range(niveles, 0, -1):                 # la familia de rectas
            v = z * k / (niveles + 1)
            ax.plot(t, (v - c[0] * t) / c[1], ls='--', lw=.9, color='gray')
        ax.plot(t, (z - c[0] * t) / c[1], lw=2.6, color='crimson',
                label=f'{c[0]:g}x₁ + {c[1]:g}x₂ = {z:.4g}')
        ax.plot(*x, 'o', ms=11, color='crimson', zorder=5)
        ax.annotate(f'({x[0]:.4g}, {x[1]:.4g})', x, textcoords='offset points',
                    xytext=(12, 9), fontsize=12, color='crimson')
    ax.set(xlim=(0, lim), ylim=(0, lim), xlabel='x₁', ylabel='x₂',
           title=titulo or f'{estado}   z = {z:.4g}' if z else titulo or estado)
    ax.grid(alpha=.25); ax.legend(loc='upper right', fontsize=8)
    return x, z, estado

print('tres funciones listas: resolver, esquinas, dibujar')


## 1 · La impresora, resuelta

El modelo de *Escribir el modelo*, tal cual, en su forma canónica:

$$
\begin{aligned}
\max_{x_1, x_2} \quad & 4x_1 + 3x_2 && \text{créditos} \\
\text{sujeto a} \quad & x_1 + x_2 \le 10 && \text{horas} \\
& 2x_1 + x_2 \le 18 && \text{polímero} \\
& x_1 + 2x_2 \le 18 && \text{energía} \\
& x_1 \ge 0,\; x_2 \ge 0
\end{aligned}
$$


In [ ]:
# === Celda 3 · El modelo, y el signo de vuelta ==================

c = np.array([4, 3])                       # créditos por filtro y por celda
A = np.array([[1, 1], [2, 1], [1, 2]])     # horas, polímero, energía
b = np.array([10, 18, 18])
recursos = ['horas', 'polímero', 'energía']

r = linprog(c=-c, A_ub=A, b_ub=b, method='highs')   # ojo: -c, porque MINIMIZA
print('plan óptimo :', r.x)
print('fun devuelto:', r.fun, '  <- viene negado')
print('créditos    :', -r.fun)

# Los assert van con isclose, NUNCA con ==: el solver da 47.99999999999999
# donde la página dice 48, y `== 48` sería False.
assert np.allclose(r.x, [8, 2]), r.x
assert np.isclose(-r.fun, 38), -r.fun
print('\ncuadra con la página 3: 8 filtros, 2 celdas, 38 créditos')


In [ ]:
# === Celda 4 · El dibujo de la página 3, hecho por la máquina ===

V = esquinas(A, b)
print('esquinas y lo que vale cada una:')
for p in V:
    print(f'   ({p[0]:.0f}, {p[1]:.0f})  ->  {p @ c:5.0f} créditos')

dibujar(c, A, b, recursos, 'la última curva de nivel que todavía toca')
plt.show()


## 2 · Mueve los precios

Aquí es donde el notebook gana a la página. **Arrastra los dos precios** y mira
lo que pasa.

El polígono no se mueve: es el mismo, siempre. Lo que gira es la recta roja, y al
girar **cambia qué esquina toca primero**.

Tres cosas que vale la pena buscar con las manos:

1. Baja el precio de la celda hasta **1**. El ganador se va a $(9,0)$: puro
   filtro.
2. Súbelo hasta **8**. Ahora gana $(2,8)$, casi pura celda.
3. Déjalo **exactamente en 4**, igual que el filtro. La recta se pone paralela a
   un lado del polígono y **empatan dos esquinas**. Ése es el caso raro de la
   página 4: infinitos óptimos, un solo valor óptimo.


In [ ]:
# === Celda 5 · Los precios, con deslizadores ====================

def probar_precios(precio_filtro=4.0, precio_celda=3.0):
    cc = np.array([precio_filtro, precio_celda])
    x, z, _ = dibujar(cc, A, b, recursos,
                      f'precios ({precio_filtro:g}, {precio_celda:g})')
    V = esquinas(A, b)
    ganan = V[np.isclose(V @ cc, z)]
    plt.show()
    print(f'valor óptimo: {z:.4g}')
    if len(ganan) > 1:
        print(f'EMPATE: {len(ganan)} esquinas alcanzan {z:.4g} ->')
        for p in ganan:
            print(f'   ({p[0]:.4g}, {p[1]:.4g})')
        print('...y todo el segmento entre ellas también. Infinitos óptimos.')
    else:
        print(f'gana una sola esquina: ({ganan[0][0]:.4g}, {ganan[0][1]:.4g})')

if HAY_WIDGETS:
    interact(probar_precios,
             precio_filtro=FloatSlider(min=1, max=10, step=.5, value=4),
             precio_celda=FloatSlider(min=1, max=10, step=.5, value=3))
else:
    probar_precios(4, 4)      # sin widgets: te enseñamos el empate


## 3 · Encuentra el umbral con la mano

*El dibujo* afirma que **el plan $(8,2)$ aguanta con 12 kWh o más**. Compruébalo
tú, de dos maneras.

Primero mueve el deslizador de la energía y mira en qué momento la respuesta deja
de ser $(8,2)$. Después corre la celda de la curva: dibuja el valor óptimo contra
la energía disponible, y **el umbral es el codo**.


In [ ]:
# === Celda 6 · La energía, con deslizador =======================

def probar_energia(energia=18.0):
    bb = np.array([10., 18., energia])
    x, z, estado = dibujar(c, A, bb, recursos, f'energía = {energia:g} kWh')
    plt.show()
    if x is None:
        print(estado); return
    igual = np.allclose(x, [8, 2])
    print(f'plan: ({x[0]:.4g}, {x[1]:.4g})   créditos: {z:.4g}')
    print('sigue siendo (8, 2)' if igual else 'YA NO es (8, 2): el umbral quedó atrás')
    print(f'gasta de energía: {A[2] @ x:.4g} de {energia:g}')

if HAY_WIDGETS:
    interact(probar_energia,
             energia=FloatSlider(min=6, max=24, step=.5, value=18))
else:
    probar_energia(11)


In [ ]:
# === Celda 7 · El umbral, dibujado =============================
# El valor óptimo como función de la energía disponible. Sube en línea recta
# mientras la energía manda, y se aplana en cuanto deja de mandar. El punto
# donde se dobla es el umbral.

es = np.linspace(4, 24, 201)
zs = [resolver(c, A, [10, 18, e])[1] for e in es]
zs = [np.nan if z is None else z for z in zs]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(es, zs, lw=2.5, color='tab:purple')
ax.axvline(12, ls='--', color='crimson')
ax.axhline(38, ls=':', color='gray')
ax.annotate('el codo: 12 kWh', (12, 30), xytext=(13.5, 24), fontsize=12,
            color='crimson', arrowprops=dict(arrowstyle='->', color='crimson'))
ax.annotate('techo 38: aquí manda otro recurso', (20, 38.6), fontsize=11,
            color='gray', ha='center')
ax.set(xlabel='energía disponible (kWh)', ylabel='créditos del mejor plan',
       title='a partir del umbral, tener más energía no sirve de nada')
ax.grid(alpha=.25)
plt.show()

print('gasto de energía del plan (8, 2):', A[2] @ np.array([8, 2]), 'kWh')


In [ ]:
# === Celda 8 · ¿Cuánto vale una unidad más de cada recurso? =====
# Se calcula RESOLVIENDO OTRA VEZ con una unidad más, que es la definición.
# No se le pide al solver: `r.ineqlin.marginals` devuelve estos mismos números
# NEGADOS, porque son los del problema que el solver minimiza.

z0 = resolver(c, A, b)[1]
x0 = resolver(c, A, b)[0]
print(f'{"recurso":10}{"gasta":>8}{"hay":>6}{"sobra":>7}{"vale 1 más":>12}\n')
precios = []
for k in range(3):
    bb = b.copy(); bb[k] += 1
    precios.append(resolver(c, A, bb)[1] - z0)
    print(f'{recursos[k]:10}{A[k] @ x0:8.0f}{b[k]:6.0f}{b[k] - A[k] @ x0:7.0f}'
          f'{precios[-1]:12.2f}')

assert np.allclose(precios, [2, 1, 0]), precios
print('\nla energía vale 0: sobran 6 kWh, y más energía no compra nada')
print('lo que devuelve el solver, para que veas la trampa:', r.ineqlin.marginals)


## 4 · Rompe el problema a propósito

*Qué es una respuesta* nombra ocho tipos de respuesta. Cuatro de ellos se pueden provocar
cambiando dos números, y conviene verlos una vez para reconocerlos después.

| Lo que provocamos | Cómo | Qué contesta el solver |
|---|---|---|
| **Óptimo único** | el modelo tal cual | un punto y un valor |
| **Empate** | poner los dos precios iguales | un punto, y se calla que hay más |
| **Infactible** | pedir 5 filtros cuando solo caben 3 piezas | se rinde: no hay región |
| **No acotado** | dejar una sola restricción que no cierra | se rinde: se puede mejorar sin fin |

Fíjate en el segundo renglón, que es el traicionero: **el solver no avisa del
empate.** Devuelve una de las dos respuestas como si fuera la única. El dibujo sí
lo enseña, y ése es el mejor argumento de la clase a favor de dibujar.


In [ ]:
# === Celda 9 · Los cuatro finales posibles =====================

casos = [
    ('óptimo único', [4, 3], A, b),
    ('empate',       [4, 4], A, b),
    ('infactible',   [1, 1], [[1, 1], [-1, 0]], [3, -5]),   # x1+x2<=3 y x1>=5
    ('no acotado',   [1, 1], [[1, -1]],         [1]),       # solo x1-x2<=1
]

fig, ejes = plt.subplots(2, 2, figsize=(12, 11))
for (nombre, cc, AA, bb), ax in zip(casos, ejes.ravel()):
    x, z, estado = dibujar(cc, AA, bb, titulo=nombre, ax=ax)
    V = esquinas(AA, bb)
    n = len(V[np.isclose(V @ np.asarray(cc, float), z)]) if z is not None else 0
    print(f'{nombre:14} -> {estado:38} '
          + (f'z = {z:.4g}, lo alcanzan {n} esquina(s)' if z is not None else ''))
plt.tight_layout(); plt.show()


---

## 5 · Ahora tú: el invernadero

Otro rincón de la nave, el mismo trabajo. **Esta historia no está resuelta en
ninguna página**: es el ejercicio.

### La situación

En la cubierta baja hay un **invernadero hidropónico**. Quedaron **4 charolas
libres** y hay que sembrarlas hoy, antes de llegar al mismo depósito.

Se pueden sembrar dos cosas, y solo dos:

- **Tubérculo**, que el depósito paga a **8 créditos** la charola.
- **Hoja**, que paga a **5**.

El tubérculo se paga mejor, así que la tentación es sembrar puro tubérculo. Otra
vez no se puede, y otra vez por lo mismo: **tres cosas se acaban**. Las horas de
**luz** de crecimiento que quedan antes del ciclo de sueño, el **sustrato** que
hay en el almacén, y el **agua** que la nave destina al invernadero.

Y los dos cultivos **no gastan lo mismo**. Uno es lento y sobrio con el agua; el
otro crece rápido y bebe el doble.

### Lo que llegó

Igual que con la impresora, nadie te da eso ordenado. Llega así:

> **Bitácora del invernadero.** Quedaron 4 charolas libres y hay que decidir qué
> sembrar antes de la parada. En el depósito pagan 8 créditos por cada charola de
> tubérculo y 5 por cada una de hoja.
>
> De luz de crecimiento nos quedan 10 horas antes del ciclo de sueño. El
> tubérculo se lleva 2 horas y la hoja 1.
>
> De sustrato hay 7 kilos, y cualquiera de los dos cultivos se lleva 1.
>
> De agua tenemos la de siempre. El tubérculo bebe 1 litro.
>
> El invernadero está a 22 grados, como toda la semana.
>
> Se me olvidaba la hoja: de agua bebe 2 litros, el doble que el tubérculo.
>
> Y el biólogo insiste en que no conviene sembrar puro tubérculo.

| | En esta historia |
|---|---|
| **Quién decide** | La tripulación, hoy, antes de llegar al depósito |
| **Qué decide** | Cuántas charolas de tubérculo y cuántas de hoja sembrar |
| **Qué lo limita** | Tres cosas que se acaban: luz, sustrato y agua |
| **Qué se quiere** | Que el total de créditos sea lo más grande posible |
| **Qué estorba** | La bitácora trae frases que no son ninguna de las cuatro anteriores |

### Las cinco cosas que hay que hacer

1. **Escribe la tabla de recursos.** Marca **qué número sobra**, **cuál falta** y
   **qué frase es ambigua**. Esta bitácora tiene una trampa de cada tipo.
2. **Escribe el modelo** recorriendo los siete pasos del lienzo.
3. **Resuélvelo y dibújalo** en la celda de abajo. Ya tienes las tres funciones.
4. **Interroga tu respuesta** con la celda de comprobación: ¿qué recurso sobra?
5. **Encuentra el umbral de tu supuesto**, con la curva del codo de la celda 7.


In [ ]:
# === Celda 10 · Tu modelo del invernadero ======================
# Está sembrado con los números DE LA IMPRESORA, no con los tuyos. Así corre de
# entrada y devuelve 38, que es visiblemente la respuesta de otro problema.
# Cámbialos por los del invernadero y vuelve a correr.

c_inv = np.array([4, 3])                          # <- créditos por charola
A_inv = np.array([[1, 1], [2, 1], [1, 2]])        # <- luz, sustrato, agua
b_inv = np.array([10, 18, 18])                    # <- de cuánto dispones
nombres_inv = ['luz', 'sustrato', 'agua']

x_inv, z_inv, estado_inv = dibujar(c_inv, A_inv, b_inv, nombres_inv,
                                   'tu invernadero')
plt.show()
print('estado :', estado_inv)
print('plan   :', x_inv)
print('valor  :', z_inv)


In [ ]:
# === Celda 11 · Interroga tu respuesta =========================
# Esto NO te dice si acertaste: te ayuda a revisar tu propio modelo, que es lo
# que vas a tener que hacer siempre.

def interrogar(c, A, b, nombres):
    x, z, estado = resolver(c, A, b)
    print('estado:', estado)
    if x is None:
        print('no hay plan que revisar; vuelve al paso 2 del lienzo'); return
    print(f'plan: ({x[0]:.4g}, {x[1]:.4g})   valor: {z:.4g}\n')
    for k, nom in enumerate(nombres):
        gasta = A[k] @ x
        sobra = b[k] - gasta
        marca = 'SE ACABA' if abs(sobra) < 1e-9 else f'sobran {sobra:.4g}'
        print(f'  {nom:10} gastas {gasta:6.4g} de {b[k]:6.4g}   {marca}')
    entero = np.allclose(x, np.round(x))
    print(f'\n¿el plan sale en piezas enteras? {"sí" if entero else "NO"}')
    if not entero:
        print('  ojo: el modelo permite fracciones y tu respuesta las usa.')
    V = esquinas(A, b)
    ganan = V[np.isclose(V @ np.asarray(c, float), z)] if len(V) else []
    print(f'¿cuántas esquinas alcanzan el valor óptimo? {len(ganan)}'
          + ('  -> hay empate: infinitos óptimos' if len(ganan) > 1 else ''))

interrogar(c_inv, A_inv, b_inv, nombres_inv)
